# Manuscript Tables and Figures

## Scientific objective
Assemble provenance, endpoint statistics, model comparisons, calibration, AD/OOD, cliffs, ablations, and workflow assets into publication-ready tables and figures without inventing missing results.

## Inputs
- All prior notebook artifacts

## Expected outputs
- `tables/manuscript_*.csv`
- `figures/manuscript_*.png`
- `reports/manuscript_artifact_index.json`

## Dependencies
pandas, matplotlib

## Reproducibility seed
`20260723`. The seed is loaded from `configs/training_config.yaml`; split files and checkpoints are persisted.

## Data and model assumptions
Only artifacts produced by executed notebooks are included. Missing analyses are listed as unavailable rather than filled with fabricated numbers.

## Validation checks
The executable cells below fail explicitly on missing/inconsistent required artifacts and save machine-readable status records.

## Interpretation of results
Interpret endpoint-level outputs only after checking prevalence, missingness, split integrity, calibration, uncertainty, and applicability-domain coverage. No notebook result is evidence that experimental toxicity testing can be replaced.

## Saved artifacts
Artifacts listed above are written under `data/`, `models/`, `results/`, `figures/`, `tables/`, or `reports/` and are consumed by later notebooks.

## Limitations
Publication formatting and journal-specific requirements remain separate from scientific validation.

## Next notebook
[README.md](./README.md)

In [1]:
from pathlib import Path
import os, json, warnings
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks directory")
os.chdir(ROOT)

from toxicity_screening.config import load_configs, execution_profile
from toxicity_screening.utils import set_global_seed, require_paths

CONFIGS = load_configs(ROOT)
PROFILE, PROFILE_CONFIG = execution_profile(CONFIGS)
SEED = int(CONFIGS["training_config"]["seed"])
set_global_seed(SEED)
print({"root": str(ROOT), "profile": PROFILE, "seed": SEED})

{'root': 'D:\\Dropbox\\Work\\Learning\\Python\\toxicity_screening_project', 'profile': 'full', 'seed': 20260723}


In [2]:
from toxicity_screening.utils import atomic_write_json
inputs={
 "dataset_registry":ROOT/"data/metadata/dataset_registry.csv",
 "endpoint_summary":ROOT/"tables/endpoint_summary.csv",
 "model_comparison":ROOT/"tables/final_model_comparison.csv",
 "calibration":ROOT/"results/calibration/calibrated_model_summary.csv",
 "ad":ROOT/"results/applicability_domain/ad_performance_summary.csv",
 "cliffs":ROOT/"results/activity_cliffs/cliff_performance.csv",
 "ablations":ROOT/"results/ablations/ablation_matrix.csv",
}
index=[]
for name,path in inputs.items():
    if path.exists():
        frame=pd.read_csv(path); destination=ROOT/f"tables/manuscript_{name}.csv"; frame.to_csv(destination,index=False); index.append({"artifact":name,"status":"included","source":str(path.relative_to(ROOT)),"output":str(destination.relative_to(ROOT))})
    else:
        index.append({"artifact":name,"status":"unavailable","source":str(path.relative_to(ROOT)),"output":None})
atomic_write_json({"profile":PROFILE,"artifacts":index,"warning":"Do not report unavailable or smoke-profile results as completed manuscript evidence."},ROOT/"reports/manuscript_artifact_index.json")
display(pd.DataFrame(index))

,artifact,status,source,output
0,dataset_registry,included,data\metadata\dataset_registry.csv,tables\manuscript_dataset_registry.csv
1,endpoint_summary,included,tables\endpoint_summary.csv,tables\manuscript_endpoint_summary.csv
2,model_comparison,included,tables\final_model_comparison.csv,tables\manuscript_model_comparison.csv
3,calibration,included,results\calibration\calibrated_model_summary.csv,tables\manuscript_calibration.csv
4,ad,included,results\applicability_domain\ad_performance_su...,tables\manuscript_ad.csv
5,cliffs,included,results\activity_cliffs\cliff_performance.csv,tables\manuscript_cliffs.csv
6,ablations,included,results\ablations\ablation_matrix.csv,tables\manuscript_ablations.csv


In [3]:
# Publication-style endpoint comparison plots.
# Matplotlib runs in a separate, stable Python process.

from pathlib import Path
import os
import subprocess
import textwrap

plot_python = Path(
    os.environ.get(
        "TOXICITY_PLOT_PYTHON",
        r"D:\Users\anaconda3\python.exe",
    )
)

if not plot_python.exists():
    raise FileNotFoundError(
        f"Plotting Python was not found: {plot_python}"
    )

worker_path = (
    ROOT
    / "reports"
    / "_notebook25_endpoint_plot_worker.py"
)

worker_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

worker_code = textwrap.dedent(
    r'''
    from pathlib import Path
    import sys

    import pandas as pd

    import matplotlib
    matplotlib.use("Agg", force=True)

    import matplotlib.pyplot as plt


    root = Path(sys.argv[1]).resolve()

    input_path = (
        root
        / "results"
        / "calibration"
        / "calibrated_model_summary.csv"
    )

    if not input_path.exists():
        raise FileNotFoundError(
            f"Calibration summary does not exist: {input_path}"
        )

    calibration = pd.read_csv(input_path)

    if "endpoint" not in calibration.columns:
        raise ValueError(
            "The calibration summary does not contain an endpoint column"
        )

    requested_metrics = [
        "test_pr_auc",
        "test_mcc",
        "test_brier",
    ]

    metric_columns = [
        metric
        for metric in requested_metrics
        if metric in calibration.columns
    ]

    if not metric_columns:
        raise ValueError(
            "None of the requested plotting columns were found. "
            f"Available columns: {list(calibration.columns)}"
        )

    figure_directory = root / "figures"
    figure_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    status_rows = []

    for metric in metric_columns:
        plot_data = calibration[
            ["endpoint", metric]
        ].copy()

        plot_data[metric] = pd.to_numeric(
            plot_data[metric],
            errors="coerce",
        )

        plot_data = plot_data.dropna(
            subset=["endpoint", metric]
        )

        if plot_data.empty:
            status_rows.append(
                {
                    "metric": metric,
                    "status": "skipped",
                    "reason": "no_valid_values",
                    "output": None,
                }
            )

            print(
                f"[PLOT] skipped metric={metric} "
                f"reason=no_valid_values",
                flush=True,
            )
            continue

        figure, axis = plt.subplots(
            figsize=(8, 4.5)
        )

        axis.bar(
            plot_data["endpoint"].astype(str),
            plot_data[metric],
        )

        axis.set_title(
            f"Endpoint comparison: {metric}"
        )

        axis.set_xlabel(
            "Toxicity endpoint"
        )

        axis.set_ylabel(
            metric
        )

        axis.tick_params(
            axis="x",
            rotation=35,
        )

        figure.tight_layout()

        output_path = (
            figure_directory
            / f"manuscript_{metric}.png"
        )

        figure.savefig(
            output_path,
            dpi=220,
            bbox_inches="tight",
        )

        plt.close(figure)

        status_rows.append(
            {
                "metric": metric,
                "status": "created",
                "reason": None,
                "output": str(
                    output_path.relative_to(root)
                ),
            }
        )

        print(
            f"[PLOT] created "
            f"metric={metric} "
            f"rows={len(plot_data)} "
            f"output={output_path}",
            flush=True,
        )

    status_path = (
        root
        / "reports"
        / "notebook25_plot_status.csv"
    )

    pd.DataFrame(status_rows).to_csv(
        status_path,
        index=False,
    )

    print(
        f"[PLOT] completed "
        f"created={sum(row['status'] == 'created' for row in status_rows)} "
        f"report={status_path}",
        flush=True,
    )
    '''
)

worker_path.write_text(
    worker_code,
    encoding="utf-8",
)

plot_environment = os.environ.copy()

plot_environment.update(
    {
        "MPLBACKEND": "Agg",
        "OMP_NUM_THREADS": "1",
        "MKL_NUM_THREADS": "1",
        "OPENBLAS_NUM_THREADS": "1",
        "NUMEXPR_NUM_THREADS": "1",
        "VECLIB_MAXIMUM_THREADS": "1",
        "BLIS_NUM_THREADS": "1",
        "PYTHONUNBUFFERED": "1",
        "PYTHONFAULTHANDLER": "1",
        "MPLCONFIGDIR": str(
            ROOT
            / "reports"
            / "_matplotlib_config"
        ),
    }
)

result = subprocess.run(
    [
        str(plot_python),
        "-X",
        "faulthandler",
        str(worker_path),
        str(ROOT),
    ],
    cwd=str(ROOT),
    env=plot_environment,
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)

    raise RuntimeError(
        "Notebook 25 plotting failed in the external process. "
        f"Exit code: {result.returncode}"
    )

generated_figures = sorted(
    (ROOT / "figures").glob(
        "manuscript_test_*.png"
    )
)

print(
    {
        "plot_python": str(plot_python),
        "figures_created": len(generated_figures),
        "figures": [
            path.name
            for path in generated_figures
        ],
    }
)

[PLOT] created metric=test_pr_auc rows=6 output=D:\Dropbox\Work\Learning\Python\toxicity_screening_project\figures\manuscript_test_pr_auc.png
[PLOT] created metric=test_mcc rows=6 output=D:\Dropbox\Work\Learning\Python\toxicity_screening_project\figures\manuscript_test_mcc.png
[PLOT] created metric=test_brier rows=6 output=D:\Dropbox\Work\Learning\Python\toxicity_screening_project\figures\manuscript_test_brier.png
[PLOT] completed created=3 report=D:\Dropbox\Work\Learning\Python\toxicity_screening_project\reports\notebook25_plot_status.csv

{'plot_python': 'D:\\Users\\anaconda3\\python.exe', 'figures_created': 3, 'figures': ['manuscript_test_brier.png', 'manuscript_test_mcc.png', 'manuscript_test_pr_auc.png']}


### Completion gate
Confirm that the declared artifacts exist before continuing to `README.md`.